# NLI Faithfulness Baseline

## Sentence-Level Evidence-Aware NLI for RAG Faithfulness Detection

This notebook implements and evaluates an NLI-based approach for detecting
whether RAG-generated answers are faithful to the retrieved context chunks.

**Model:** MoritzLaurer/mDeBERTa-v3-base-mnli-xnli

**Approach:** Sentence-level evidence aggregation with multiple strategies

---

## Table of Contents

1. Environment and Reproducibility
2. Data Loading and Checks
3. Model Loading
4. Small Smoke Test
5. Validation Inference
6. Aggregation Comparison
7. Threshold Selection
8. Test Evaluation
9. TF-IDF Comparison
10. Evidence and Error Analysis
11. Limitations

---

## 1. Environment and Reproducibility

In [ ]:
import sys
import os

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from nli_faithfulness import (
    DEFAULT_MODEL_NAME,
    CHUNK_COLUMNS,
    RESULTS_DIR,
    CACHE_DIR,
)

print(f"Model: {DEFAULT_MODEL_NAME}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Cache dir: {CACHE_DIR}")

## 2. Data Loading and Checks

In [ ]:
from nli_faithfulness.data import load_dataset, get_stratified_sample, dataset_summary

# Load all splits
train, val, test = load_dataset()

print("=== Dataset Summary ===")
print(f"\nTrain: {dataset_summary(train)}")
print(f"\nVal: {dataset_summary(val)}")
print(f"\nTest: {dataset_summary(test)}")

## 3. Model Loading

In [ ]:
from nli_faithfulness.inference import NLIModel

# Load NLI model
model = NLIModel(model_name=DEFAULT_MODEL_NAME)

print(f"\nDevice: {model.device}")
print(f"Max length: {model.max_length}")
print(f"Label mapping: {model.id2label}")

# Save model info
model.save_model_info()

## 4. Small Smoke Test

In [ ]:
from nli_faithfulness.segmentation import split_answer_into_units, segment_dataset

# Create stratified sample for smoke test
smoke_sample = get_stratified_sample(val, n_per_class=10, seed=42)

print(f"Smoke test sample: {len(smoke_sample)} samples")

# Segment all answers
smoke_segments = segment_dataset(smoke_sample.samples)

# Show example
example_case = list(smoke_segments.keys())[0]
example_seg = smoke_segments[example_case]
print(f"\nExample case: {example_case}")
print(f"Number of sentence units: {len(example_seg)}")
for unit in example_seg:
    print(f"  Sentence {unit.sentence_id}: {unit.text[:80]}...")

In [ ]:
# Run NLI inference on smoke test
from nli_faithfulness.inference import batch_inference

# Note: First run will download the model
smoke_cache = CACHE_DIR / "smoke_test_scores.csv"
smoke_scores = batch_inference(
    model=model,
    samples=smoke_sample.samples,
    segments=smoke_segments,
    batch_size=4,
    cache_path=smoke_cache,
)

print(f"\nSmoke test scores shape: {smoke_scores.shape}")
print(f"Total pairs: {len(smoke_scores)}")

In [ ]:
# Verify probability sums
prob_sums = (
    smoke_scores['p_entailment'] + 
    smoke_scores['p_neutral'] + 
    smoke_scores['p_contradiction']
)
print(f"Probability sum stats:")
print(f"  Min: {prob_sums.min():.6f}")
print(f"  Max: {prob_sums.max():.6f}")
print(f"  Mean: {prob_sums.mean():.6f}")

# Check for truncation
print(f"\nTruncation check: See window_id=0 always (no truncation in smoke test)")

In [ ]:
# Show evidence example
from nli_faithfulness.aggregation import get_best_evidence

example_evidence = get_best_evidence(smoke_scores, example_case)
print(f"Best evidence for case {example_case}:")

if 'best_entailment' in example_evidence:
    ent = example_evidence['best_entailment']
    print(f"\n  Best entailment:")
    print(f"    Chunk: {ent['chunk_id']}, Window: {ent['window_id']}")
    print(f"    P(entailment): {ent['p_entailment']:.4f}")
    print(f"    Hypothesis: {ent['hypothesis'][:100]}...")

if 'best_contradiction' in example_evidence:
    cont = example_evidence['best_contradiction']
    print(f"\n  Best contradiction:")
    print(f"    Chunk: {cont['chunk_id']}, Window: {cont['window_id']}")
    print(f"    P(contradiction): {cont['p_contradiction']:.4f}")
    print(f"    Hypothesis: {cont['hypothesis'][:100]}...")

## 5. Validation Inference (Full Run)

*This section will be executed after smoke test passes.*

In [ ]:
# Uncomment to run full validation
# val_cache = CACHE_DIR / "validation_scores.csv"
# val_scores = batch_inference(
#     model=model,
#     samples=val.samples,
#     segments=segment_dataset(val.samples),
#     batch_size=8,
#     cache_path=val_cache,
# )

## 6. Aggregation Comparison

*Executed after validation inference completes.*

In [ ]:
# Compare aggregation strategies
# from nli_faithfulness.aggregation import compare_strategies, AGGREGATION_STRATEGIES
#
# # Get ground truth
# y_true = pd.Series({s.case_id: int(s.binary_faithfulness) for s in val})
#
# # Compare strategies
# strategies = list(AGGREGATION_STRATEGIES.keys())
# comparison = compare_strategies(val_scores, strategies, y_true)
# print(comparison.to_string())

## 7. Threshold Selection

*Executed after strategy comparison.*

In [ ]:
# Threshold selection (per strategy)
# from nli_faithfulness.evaluation import find_best_threshold
#
# best_thresholds = {}
# for strategy in strategies:
#     preds = apply_aggregation_strategy(val_scores, strategy)
#     merged = preds.merge(y_true.reset_index().rename(...), on='case_id')
#     threshold, metrics = find_best_threshold(
#         merged['binary_faithfulness'].values,
#         merged['faithfulness_score'].values
#     )
#     best_thresholds[strategy] = {'threshold': threshold, 'metrics': metrics}

## 8. Test Evaluation

*Executed once after validation tuning is complete.*

In [ ]:
# Test evaluation (only once!)
# test_scores = batch_inference(
#     model=model,
#     samples=test.samples,
#     segments=segment_dataset(test.samples),
#     batch_size=8,
#     cache_path=CACHE_DIR / "test_scores.csv",
# )

## 9. TF-IDF Comparison

*Compare with existing TF-IDF baseline.*

In [ ]:
# Load TF-IDF results
# from nli_faithfulness.evaluation import load_tfidf_results, compare_with_tfidf
#
# tfidf_results = load_tfidf_results()
# print("TF-IDF Baseline Results:")
# print(json.dumps(tfidf_results.get('faithfulness_test', {}), indent=2))

## 10. Evidence and Error Analysis

*Analyze predictions and errors.*

In [ ]:
# Evidence extraction
# from nli_faithfulness.aggregation import get_best_evidence
#
# # Get evidence for all test samples
# evidence_records = []
# for case_id in test_scores['case_id'].unique():
#     ev = get_best_evidence(test_scores, case_id)
#     if ev:
#         evidence_records.append({
#             'case_id': case_id,
#             **ev.get('best_entailment', {}),
#             **ev.get('best_contradiction', {})
#         })
#
# evidence_df = pd.DataFrame(evidence_records)

## 11. Limitations

1. **Sentence-level granularity:** Answer sentences are not necessarily atomic claims.
   A single sentence may contain multiple claims, or a claim may span multiple sentences.

2. **Entailment ≠ Full support:** NLI entailment means the hypothesis is compatible
   with the premise, not necessarily that all facts are verified.

3. **Neutral ambiguity:** NLI neutral can mean either:
   - The premise doesn't address the hypothesis, OR
   - The hypothesis contains information not in the premise
   These have very different implications for faithfulness.

4. **No claim-level reasoning:** The current approach cannot identify which specific
   claim in an answer is not supported.

5. **Length mismatch:** Long chunks may contain information not directly relevant
   to the specific answer sentence, leading to false entailments.

---